In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

initial_path = "/Volumes/courier_delivery/source/input_files/dbt_orders_initial_100.csv"

incremental_path = "/Volumes/courier_delivery/source/input_files/dbt_orders_incremental_50.csv"

initial_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(initial_path)
)

incremental_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(incremental_path)
)

print("Initial records:", initial_df.count())
print("Incremental records:", incremental_df.count())

display(initial_df.limit(5))
display(incremental_df.limit(5))

In [0]:
print(initial_df.columns)
print(incremental_df.columns)

In [0]:
initial_bronze = (
    initial_df
    .withColumn("batch_id", lit("BATCH_001"))
    .withColumn("source_file", lit("dbt_orders_initial_100.csv"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

incremental_bronze = (
    incremental_df
    .withColumn("batch_id", lit("BATCH_002"))
    .withColumn("source_file", lit("dbt_orders_incremental_50.csv"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

In [0]:
initial_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("courier_delivery.bronze.orders_initial_raw")

incremental_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("courier_delivery.bronze.orders_incremental_raw")

In [0]:
%sql
SELECT * FROM courier_delivery.bronze.orders_initial_raw;

In [0]:
%sql
SELECT * FROM courier_delivery.bronze.orders_incremental_raw;

In [0]:
%sql
SELECT 'INITIAL' AS batch, COUNT(*) AS total_records
FROM courier_delivery.bronze.orders_initial_raw

UNION ALL

SELECT 'INCREMENTAL', COUNT(*)
FROM courier_delivery.bronze.orders_incremental_raw;

In [0]:
# Import functions required for data validation
from pyspark.sql.functions import col, when, lit, concat_ws

# Define validation conditions for the initial batch
initial_checked = initial_bronze.withColumn(
    "rejection_reason",

    # Collect all validation failures into one column
    concat_ws(
        ", ",

        # Check whether the order ID is missing
        when(col("order_id").isNull(), lit("Missing order_id")),

        # Check whether the customer ID is missing
        when(col("customer_id").isNull(), lit("Missing customer_id")),

        # Check whether the product ID is missing
        when(col("product_id").isNull(), lit("Missing product_id")),

        # Check whether the quantity is zero or negative
        when(col("quantity").isNull() | (col("quantity") <= 0), lit("Invalid quantity")),

        # Check whether the unit price is missing or negative
        when(col("unit_price").isNull() | (col("unit_price") < 0), lit("Invalid unit_price")),

        # Check whether the sales amount is missing or negative
        when(col("sales_amount").isNull() | (col("sales_amount") < 0), lit("Invalid sales_amount"))
    )
)

# Apply the same validation rules to the incremental batch
incremental_checked = incremental_bronze.withColumn(
    "rejection_reason",
    concat_ws(
        ", ",
        when(col("order_id").isNull(), lit("Missing order_id")),
        when(col("customer_id").isNull(), lit("Missing customer_id")),
        when(col("product_id").isNull(), lit("Missing product_id")),
        when(col("quantity").isNull() | (col("quantity") <= 0), lit("Invalid quantity")),
        when(col("unit_price").isNull() | (col("unit_price") < 0), lit("Invalid unit_price")),
        when(col("sales_amount").isNull() | (col("sales_amount") < 0), lit("Invalid sales_amount"))
    )
)

# Separate valid and rejected initial records
initial_valid = initial_checked.filter(col("rejection_reason") == "")

# Keep records that failed at least one validation
initial_rejected = initial_checked.filter(col("rejection_reason") != "")

# Separate valid and rejected incremental records
incremental_valid = incremental_checked.filter(col("rejection_reason") == "")

# Keep incremental records that failed at least one validation
incremental_rejected = incremental_checked.filter(col("rejection_reason") != "")

# Display the record counts for both batches
print("Initial valid:", initial_valid.count())
print("Initial rejected:", initial_rejected.count())

print("Incremental valid:", incremental_valid.count())
print("Incremental rejected:", incremental_rejected.count())

In [0]:
# Combine rejected records from both batches into one DataFrame
all_rejected = initial_rejected.unionByName(incremental_rejected)

# Save rejected records as a Delta table in the Bronze schema
all_rejected.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("courier_delivery.bronze.orders_rejects")

# Display rejected records with their rejection reasons
display(all_rejected)

In [0]:
# Create an audit record for the initial batch
initial_audit = spark.createDataFrame(
    [
        ("BATCH_001", "INITIAL", initial_bronze.count(),
         initial_valid.count(), initial_rejected.count())
    ],
    ["batch_id", "batch_type", "total_records", "valid_records", "rejected_records"]
)

# Create an audit record for the incremental batch
incremental_audit = spark.createDataFrame(
    [
        ("BATCH_002", "INCREMENTAL", incremental_bronze.count(),
         incremental_valid.count(), incremental_rejected.count())
    ],
    ["batch_id", "batch_type", "total_records", "valid_records", "rejected_records"]
)

# Combine both audit records into one DataFrame
batch_audit_df = initial_audit.unionByName(incremental_audit)

# Add the timestamp when the audit information is created
batch_audit_df = batch_audit_df.withColumn(
    "audit_timestamp", current_timestamp()
)

# Save the audit information as a Delta table
batch_audit_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("courier_delivery.bronze.batch_audit")

# Display the batch audit report
display(batch_audit_df)

In [0]:

# Display all rejected initial records and their rejection reasons
display(
    initial_rejected.select(
        "order_id",
        "customer_id",
        "product_id",
        "quantity",
        "unit_price",
        "sales_amount",
        "rejection_reason"
    )
)

In [0]:
# Count how many records were rejected for each reason
reason_counts = (
    initial_rejected
    .groupBy("rejection_reason")
    .count()
    .orderBy(col("count").desc())
)

# Display rejection reason summary
display(reason_counts)

In [0]:
# Find order IDs that appear more than once in the initial batch
duplicate_orders = (
    initial_bronze
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

# Display duplicate order IDs
display(duplicate_orders)

# Count the number of duplicate order IDs
print("Duplicate order IDs:", duplicate_orders.count())

In [0]:
# Display all rejected records with their individual rejection reasons
display(
    initial_rejected.select(
        "order_id",
        "customer_id",
        "product_id",
        "quantity",
        "unit_price",
        "sales_amount",
        "rejection_reason"
    ).orderBy("order_id")
)

# Count the actual number of rejected rows
print("Total rejected rows:", initial_rejected.count())

# Count the number of unique rejected order IDs
print(
    "Unique rejected order IDs:",
    initial_rejected.select("order_id").distinct().count()
)

In [0]:

# Import the functions needed for data cleaning and type conversion
from pyspark.sql.functions import col, trim, upper, to_date
from pyspark.sql.types import IntegerType, DecimalType

# Select only records that passed Bronze validation
silver_initial = initial_valid

# Remove the rejection_reason column because these records are valid
silver_initial = silver_initial.drop("rejection_reason")

# Remove unwanted spaces and standardise text columns
silver_initial = silver_initial.withColumn("city", upper(trim(col("city"))))
silver_initial = silver_initial.withColumn("payment_method", upper(trim(col("payment_method"))))
silver_initial = silver_initial.withColumn("order_status", upper(trim(col("order_status"))))

# Convert order_date into a proper date format
silver_initial = silver_initial.withColumn("order_date", to_date(col("order_date")))

# Convert quantity into an integer
silver_initial = silver_initial.withColumn("quantity", col("quantity").cast(IntegerType()))

# Convert unit_price, discount_pct and sales_amount into decimal values
silver_initial = silver_initial.withColumn("unit_price", col("unit_price").cast(DecimalType(12, 2)))
silver_initial = silver_initial.withColumn("discount_pct", col("discount_pct").cast(DecimalType(5, 2)))
silver_initial = silver_initial.withColumn("sales_amount", col("sales_amount").cast(DecimalType(12, 2)))

# Remove duplicate order IDs, keeping one record per order
silver_initial = silver_initial.dropDuplicates(["order_id"])

# Display the cleaned initial records
display(silver_initial)

# Print the final number of initial records
print("Initial Silver record count:", silver_initial.count())


In [0]:

# Save the cleaned initial records as a Delta table in the Silver schema
silver_initial.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("courier_delivery.silver.orders")

# Confirm that the Silver table was created
display(spark.table("courier_delivery.silver.orders"))

# Verify the number of records saved
print(
    "Records in Silver table:",
    spark.table("courier_delivery.silver.orders").count()
)
